Last notebook: the goal is to create an unique pipeline to augment the entire dataset.

In [ ]:
import os
import re
import sys
import copy
import json
import time
import random
import inflect

from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv
from num2words import num2words

from utils.support_functions import load_dataset, save_cache, check_cnl, load_cache

In [ ]:
# note: the template for this file is available. Please copy your OpenAI API key to run this script.
! source SET_KEY.sh

load_dotenv(".env") ;

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    sys.exit("OpenAI Key Not Set.")

In [ ]:
############################################################
######################### SETTINGS #########################
############################################################


JSON_FILENAME = "NL2CNL_reconstructed" #initial set of rules

JSON_FILE_INPUT = f"./{JSON_FILENAME}.json"

MODEL = 'gpt-5-mini'
CNL_ENDPOINT = "http://160.97.63.29:3003/api/compile"


CACHE_FILENAME = f"{JSON_FILENAME}_synonyms_cache.json"
AUGMENTED_FILENAME = f"{JSON_FILENAME}_augmented.json"
REPHRASE_FILENAME = f"{AUGMENTED_FILENAME.replace('.json', '')}_rephrased.json"



REPH_FACTS = 1
REPH_RULES = 3

SYSTEM_PROMPT_SYNONYMS = {
 "role": "system",
  "content": '''
  ### ROLE
You are an NLP Data Augmentation engine specialized in semantic mapping between Controlled Natural Language (CNL) and Natural Language (NL).

### TASK
Generate lexical alternatives (synonyms) for the tokens specified in the Linkings dictionary, maintaining consistency between the formal representation (CNL) and the natural representation (NL).

### STRICT FORMATTING RULES
1. Respond EXCLUSIVELY with a valid JSON object.
2. Do NOT include preamboles, explanations, or conversational filler (e.g., "Sure, here is...", "Certainly").
3. Do NOT use markdown code blocks (avoid ```json).
4. The response must start with '{' and end with '}'.
5. CNL tokens MUST NOT contain spaces. Any space in a CNL synonym MUST be replaced with an underscore (_).
6. Synonyms should be concise and consist of a maximum of TWO words.
7. Strictly adhere to the JSON schema described below.

### JSON OUTPUT SCHEMA
{
  "mapping": [
    {
      "original": {"cnl": "string", "nl": "string"},
      "synonyms": [
        {"cnl": "string", "nl": "string"},
        {"cnl": "string", "nl": "string"},
        {"cnl": "string", "nl": "string"}
      ]
    }
  ]
}

### EXAMPLE (FEW-SHOT)
Input NL: "The system must start the engine."
Input CNL: "system shall start engine"
Linkings: {"start": "start"}

Output:
{"mapping":[{"original":{"cnl":"start","nl":"start"},"synonyms":[{"cnl":"activate","nl":"activate"},{"cnl":"run","nl":"run"},{"cnl":"trigger","nl":"trigger"}]}]}

'''}


SYSTEM_PROMPT_REPHRASE = {
 "role": "system",
  "content": '''
You are a conditional linguistic rephrasing engine. Your goal is to process a single input sentence and return a JSON object containing the requested variations.

**Processing Rules:**
1. **Protected Terms:** All items in the provided "Protected Terms" list must be preserved in the rephrasings. You can **not** remove them. They can be made plural if grammatically necessary, but the root must remain.
2. **Count Logic:** Generate the **exact** number of rephrasings specified by the "Target Count".
3. **Variety Requirement:** Mere punctuation or capitalization changes are strictly forbidden; you must substantially alter the syntactic structure or vocabulary (excluding protected terms).

### STRICT FORMATTING RULES
1. Respond EXCLUSIVELY with a valid JSON object.
2. Do NOT include preambles, explanations, or conversational filler.
3. Do NOT use markdown code blocks (avoid ```json).
4. The response must start with `{` and end with `}`.
5. Strictly adhere to the JSON schema described below.

**JSON Schema:**
{
  "rephrasings": [
    "String variation 1",
    "String variation 2",
    ...
  ]
}

**Few-Shot Example:**
Input:
Source Sentence: "The data flows to the server."
Protected Terms: ["server"]
Target Count: 3

Output:
{
  "rephrasings": [
    "To the server, the data flows.",
    "The server receives the flow of data.",
    "Data is transmitted towards the server."
  ]
}

**Task:**

'''}



openai_client = OpenAI()

## Cache generation

In [ ]:
def generate_synonyms(CNL, NL, linking, client=openai_client, system_promt=SYSTEM_PROMPT_SYNONYMS):
    user_content = f"""
    "CNL": "{CNL}",
    "NL": "{NL}",
    "Linking": {json.dumps(linking)}
    """

    response = client.chat.completions.create(
        model=MODEL,  
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": system_promt["content"],
            },
            {
                "role": "user",
                "content": user_content,
            },
        ],
    )

    return response.choices[0].message.content

def generate_synonyms_batch(dataset, cache):
    if cache is None:
        cache = {}

    for r in tqdm(dataset):

        cnl = r.get("CNL_V2", "")
        nl = r.get("NL_V2", "")
        linking = r.get("linking", {})

        missing_for_this_sentence = {}
        for cnl_token, nl_token in linking.items():
            cache_key = (cnl_token.lower(), nl_token.lower())
            if cache_key not in cache:
                missing_for_this_sentence[cnl_token] = nl_token

        if not missing_for_this_sentence:
            continue

        try:
            rs = generate_synonyms(cnl, nl, missing_for_this_sentence)

            # clean_rs = rs.strip().replace("```json", "").replace("```", "") # not required, forced json response format with openai
            out = json.loads(rs)

            if "mapping" in out:
                for item in out["mapping"]:
                    cnl_t = item["original"]["cnl"]
                    nl_t = item["original"]["nl"]
                    synonyms = item["synonyms"]

                    cache_key = (cnl_t.lower(), nl_t.lower())
                    cache[cache_key] = synonyms
            else:
                print("Format Error")

        except json.JSONDecodeError:
            print("Json format error")
        except Exception as e:
            print(f"Generic error: {e}")


In [ ]:
ds = load_dataset(JSON_FILE_INPUT)

In [ ]:
cached_elems = load_cache(CACHE_FILENAME)
EMPTY_CACHE = False

if cached_elems == {}:
    print("No cache found, starting from empty cache.")
    EMPTY_CACHE = True

In [ ]:
if EMPTY_CACHE:
    generate_synonyms_batch(ds, cached_elems)

In [ ]:
print(f" {len(cached_elems)} unique synonyms cached.")

In [ ]:
if EMPTY_CACHE:
    save_cache(cached_elems, CACHE_FILENAME)

## Data Augmentation

In [ ]:
def get_random_permutations(linking, n=3, cache_size = 3):
    permutations = []
    
    card = len(linking)
    if card == 1: 
        return [[x] for x in range(n)]
    while len(permutations) < n:
        p = [random.randint(0,cache_size - 1) for _ in range(card)]
        if p not in permutations:
            permutations.append(p)
    return permutations

def get_replacements(linking, permutations, cache):
    if cache is None:
        cache = {}
        
    out = {}
    words = list(linking.items())

    for p_idx, perm in enumerate(permutations):
        out[p_idx] = {}
        
        for w_idx, (cnl_token, nl_token) in enumerate(words):
            cache_key = (cnl_token.lower().replace(" ", "_"), nl_token.lower())
            
            if cache_key in cache:                
                syn_idx = perm[w_idx]
                out[p_idx][cache_key] = cache[cache_key][syn_idx]


            else:
                out[p_idx][cache_key] = {
                    "cnl": cnl_token.replace(" ", "_").lower(), 
                    "nl": nl_token.lower()
                }
    
    return out

In [ ]:
# functions to add randomness 

def _build_number_words_dict():
    """Build a dictionary of number words (0-30) excluding those with hyphens."""
    num_dict = {}
    for i in range(2,31):  # 0-30
        word = num2words(i)
        
        num_dict[word.replace("-", "")] = i
    return num_dict

NUMBER_WORDS = _build_number_words_dict()

def get_variables(asp):
    pattern = r'(?<=.)[A-Z][0-9]*'
    variables = set(re.findall(pattern, asp))
    return variables

def replace_variables(text, variables_map, is_cnl = True):
    if is_cnl:
        predicates = text.split(".")[:-2]
        rule = text.split(".")[-2] + "."
    else:
        predicates = []
        rule = text
    pattern = r'(?<=.)[A-Z][0-9]*\W'
    def replacer(match):
        matched = match.group(0)
        key = matched[:-1]
        suffix = matched[-1]
        return variables_map.get(key, key) + suffix
    
    return ".".join(predicates + [re.sub(pattern, replacer, rule)])

def introduce_variability(nl, cnl, variables, replace_numbers=True):
    
    forbidden_letters = set()
    for var in variables:
        for char in var:
            if char.isupper():
                forbidden_letters.add(char)

    available_letters = [chr(i) for i in range(65, 91) if chr(i) not in forbidden_letters]

    sorted_vars = sorted(variables, key=len, reverse=True)
    var_mapping = {}

    for var in sorted_vars:
        replacement = random.choice(available_letters)
        available_letters.remove(replacement)
        var_mapping[var] = replacement

    edited_nl = False

    print(f"Variable mapping: {var_mapping}")
    if var_mapping: # if there are subs to do 
        cnl = replace_variables(cnl, var_mapping, is_cnl=True)
        print(f"processed: {cnl}")
        if get_variables(nl) != set(): # also if nl has vars 
            nl = replace_variables(nl, var_mapping, is_cnl=False)
            edited_nl = True
    
    if replace_numbers:
        # Helper function to extract all numbers from text (returns set of integer values)
        def extract_numbers_from_text(text):
            numbers = set()
            for digit_str in re.findall(r'\b\d+\b', text):
                if (digit_int := int(digit_str)) in NUMBER_WORDS.values():
                    numbers.add(digit_int)            
            for word_num in NUMBER_WORDS.keys():
                pattern = r'\b' + re.escape(word_num) + r'\b'
                if re.search(pattern, text, re.IGNORECASE):
                    numbers.add(NUMBER_WORDS[word_num])
            
            return numbers
        
        # Extract numbers from NL and CNL
        nl_numbers = extract_numbers_from_text(nl)
        cnl_numbers = extract_numbers_from_text(cnl)

        print(f"Numbers in NL: {nl_numbers}")
        print(f"Numbers in CNL: {cnl_numbers}")
        
        # Only keep numbers that appear in BOTH
        common_numbers = nl_numbers & cnl_numbers
        
        print(common_numbers)
        if common_numbers:
            # Sort to maintain relative order
            sorted_common_numbers = sorted(common_numbers)
            
            # Generate random replacements for each common number (without hyphens, and all unique)
            random_replacements = set()
            while len(random_replacements) < len(sorted_common_numbers):
                new_num = random.randint(1, 30)
                new_word = num2words(new_num)
                if '-' not in new_word:
                    random_replacements.add(new_num)
            
            # Sort the random replacements to maintain relative order
            sorted_random = sorted(list(random_replacements))
            
            # Create mapping: original value ---> new random value (with order preserved)
            value_mapping = {old_val: new_val for old_val, new_val in zip(sorted_common_numbers, sorted_random)}
            
            # Create replacements for both digit and word forms
            num_replacements = {}
            
            # Add digit form replacements
            for match in re.finditer(r'\b\d+\b', nl + ' ' + cnl):
                digit_str = match.group(0)
                digit_int = int(digit_str)
                if digit_int in value_mapping:
                    new_num = value_mapping[digit_int]
                    num_replacements[digit_str] = (str(new_num), 'digit')
            
            # Add word form replacements
            combined_text = nl + ' ' + cnl
            for word_num, digit_val in NUMBER_WORDS.items():
                if digit_val in value_mapping:
                    pattern = r'\b' + re.escape(word_num) + r'\b'
                    match = re.search(pattern, combined_text, re.IGNORECASE)
                    if match:
                        actual_word = match.group(0)
                        new_num = value_mapping[digit_val]
                        new_word = num2words(new_num)
                        num_replacements[actual_word] = (new_word, 'word')
            
            # Replace numbers by length (longest first, need to sort them)
            for old_num in sorted(num_replacements.keys(), key=len, reverse=True):
                new_value, num_type = num_replacements[old_num]
                if num_type == 'digit':
                    nl = nl.replace(old_num, new_value)
                    cnl = cnl.replace(old_num, new_value)
                # Case-insensitive replacement for words
                else:
                    nl = re.sub(r'\b' + re.escape(old_num) + r'\b', new_value, nl, flags=re.IGNORECASE)
                    cnl = re.sub(r'\b' + re.escape(old_num) + r'\b', new_value, cnl, flags=re.IGNORECASE)
    
    return (nl, cnl, edited_nl)


def apply_variants(item):
    a,b,c = introduce_variability(item["NL_V2"], item["CNL_V2"], get_variables(item["ASP"]), item["Category"] == "Definition Const/Compound")
    print(a,b,c)
    item["NL_V2"], item["CNL_V2"], item["replaced_nl"] = a,b,c
    item["ASP"] = newAsp if (newAsp := check_cnl(item["CNL_V2"], CNL_ENDPOINT)) != '' else "ERROR"
    return item

In [ ]:
# test to assert behaviour

i =   {
            "ASP": ":- level_vertex(L,V), parent_relation(P,V), not level_vertex(L-1,P).",
            "CNL_V2": "A level_vertex is identified by a value, by an id. A parent_relation is identified by a first vertex, by a second vertex. It is prohibited that there is a level_vertex with value L, with id V, whenever there is a parent_relation with first vertex P, with second vertex V, whenever there is not a level_vertex with value L-1, with id P.",
            "NL_V2": "It is prohibited that a level vertex at level L has a parent relation from another vertex unless that other vertex is at level L-1.",
            "Category": "Negative Strong Constraint",
            "linking": {
                "level_vertex": "level vertex",
                "parent_relation": "parent relation"
            },
            "Id": "VAR_00377_1",
        }
 
print(apply_variants(i))

In [ ]:
p_eng = inflect.engine()

def smart_replace(text, old_word, new_word):

    def replace_keep_case(match):
        matched_str = match.group(0)
        if matched_str and matched_str[0].isupper():
            return new_word.capitalize()
        return new_word.lower()

    pattern = re.compile(r'\b' + re.escape(old_word) + r'\b', re.IGNORECASE)
    return pattern.sub(replace_keep_case, text)

def dataset_expansion(original_entry, replacing_set):
    expanded_variants = []

    for i in range(len(replacing_set)):
        variant = copy.deepcopy(original_entry)
        current_replacements = replacing_set.get(i, {})
        new_linking = {}
        
        for (old_cnl, old_nl), new_pair in current_replacements.items():
            new_cnl = new_pair['cnl']
            new_nl = new_pair['nl']

            old_nl_plural = p_eng.plural(old_nl)
            new_nl_plural = p_eng.plural(new_nl)
            
            old_cnl_plural = p_eng.plural(old_cnl).upper()
            new_cnl_plural = p_eng.plural(new_cnl).upper()
            
            variant["CNL_V2"] = smart_replace(variant["CNL_V2"], old_cnl_plural, new_cnl_plural)
            variant["NL_V2"] = smart_replace(variant["NL_V2"], old_nl_plural, new_nl_plural)

            variant["CNL_V2"] = smart_replace(variant["CNL_V2"], old_cnl, new_cnl)
            variant["NL_V2"] = smart_replace(variant["NL_V2"], old_nl, new_nl)

            new_linking[new_cnl.lower()] = new_nl.lower()
        
        variant["linking"] = new_linking
        variant["generated"] = True
        
        try:
            variant = apply_variants(variant)
            if variant["ASP"] == 'ERROR':
                variant["ASP"] = "ERROR" 
        except NameError:
            variant["ASP"] = "ERROR"

        base_id = original_entry['Id'].split('_')[1] if '_' in original_entry['Id'] else original_entry['Id']
        variant["Id"] = f"VAR_{base_id}_{i+1}"
        
        expanded_variants.append(variant)

    return expanded_variants

In [ ]:
AUGMENTATION_REQUIRED = True

if os.path.exists(AUGMENTED_FILENAME):
    print(f"Augmented dataset {AUGMENTED_FILENAME} already exists.")
    AUGMENTATION_REQUIRED = False

In [ ]:
if AUGMENTATION_REQUIRED:

    out = []
    errors = []

    for i in tqdm(range(len(ds))):
        
        point = ds[i]
        point['Id'] = f"ORG_{i:05d}"
        point['generated'] = False
        out.append(point)
        replacing_set = get_replacements(point['linking'],get_random_permutations(point['linking']), cached_elems)
        for d in dataset_expansion(point, replacing_set):
            if d['ASP'] != "ERROR":
                out.append(d)
            else:
                errors.append(d)

In [ ]:
if AUGMENTATION_REQUIRED:
    print("~" * 30)
    print(f"Generated {len(out) - len(ds)} variants.")
    print(f"Generated {len(errors)} variants with errors.")
    print(f"total variants: {len(out)}")
    print("~" * 30)

In [ ]:
if AUGMENTATION_REQUIRED:
    try:
        with open(AUGMENTED_FILENAME, 'w', encoding='utf-8') as f:
            json.dump( {"data_dict" : out}, f, indent=4, ensure_ascii=False)
        print(f"File saved at {AUGMENTED_FILENAME}")
        with open(f"{JSON_FILENAME}_augmented_errors.json", 'w', encoding='utf-8') as f:
            json.dump( {"data_dict" : errors}, f, indent=4, ensure_ascii=False)
        print(f"File saved at {JSON_FILENAME}_augmented_errors.json")
    except Exception as e:
        print(f"Error: {e}")

## Rephrasing

In [ ]:
def generate_rephrase(data, client=openai_client):
    user_content = f"""
    Source Sentence: {data["NL_V2"]}
    Protected Terms: {data["linking"]}
    Target Count: {data["count"]}
    
    Return the result strictly as a JSON object.
    """

    response = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"}, 
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT_REPHRASE["content"],
            },
            {
                "role": "user",
                "content": user_content,
            },
        ],
    )

    return response.choices[0].message.content


In [ ]:
ds = load_dataset(AUGMENTED_FILENAME)

In [ ]:
MAX_RETRIES = 3
RETRY_DELAY = 0.1


out = {"data_dict" : []}
final_data = {"data_dict" : []}

for i in tqdm(range(0, len(ds)), desc=f"Rephrasing sentences"):

    ds[i]["Rephrased"] = False
    final_data["data_dict"].append(ds[i])
    
    to_rephrase = {"NL_V2": ds[i]["NL_V2"], "linking": list(ds[i]["linking"].values()), "count": REPH_FACTS if ds[i]['Category'] == "Definition Whenever" else REPH_RULES}
    
#        Source Sentence: {data["NL_V2"]}
#        Protected Terms: {data['linking']}
#        Target Count: {data["count"]}
    final_out = None
    

    # need 2 retry for spurious json errs 
    for attempt in range(MAX_RETRIES):
        try:
            data = generate_rephrase(to_rephrase)
            
            # clean_rs = data.strip().replace("```json", "").replace("```", "") # NOT required, forced json response format with openai
            
            final_out = json.loads(data)
            
            if "rephrasings" not in final_out:
                raise ValueError("Wrong json format.")
                
            break 
            
        except (json.JSONDecodeError, ValueError) as e: # errori da json loads mannaiafuefekjwnfioeb
            print(f"[Batch {i}] JSON Error (Attempt {attempt + 1}/{MAX_RETRIES}): {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_DELAY)
            else:
                print(f"! Batch {i} attempted {MAX_RETRIES} times. Skipping due to repeated invalid json.")
                final_out = None
    
    if final_out is not None: # if i had no prob with json parse 
        
        # save that to sentences 
        out["data_dict"].append({
            "Id": ds[i]["Id"],
            "Original": ds[i]["NL_V2"],
            "Rephrasings": final_out.get("rephrasings", [])
            })

        for idx, rephr in enumerate(final_out["rephrasings"]):
                newItem = ds[i].copy() 
                newItem["NL_V2"] = rephr
                newItem["Id"] += f"_R{idx}"
                newItem["Rephrased"] = True
                final_data["data_dict"].append(newItem)

In [ ]:
print(f"Total dataset size: {len(final_data['data_dict'])}")

In [ ]:
try:
    fl = f"{REPHRASE_FILENAME.replace('.json', '')}_sentences.json"
    with open(fl, 'w', encoding='utf-8') as f:
        json.dump( out, f, indent=4, ensure_ascii=False)
    print(f"File saved at {fl}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
try:
    with open(REPHRASE_FILENAME, 'w', encoding='utf-8') as f:
        json.dump( final_data, f, indent=4, ensure_ascii=False)
    print(f"File saved at {REPHRASE_FILENAME}")
except Exception as e:
    print(f"Error: {e}")